# The Cooling Gap — the notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/john-bruene/pointzerofive-data/blob/main/the-cooling-gap/story_003_cooling_gap.ipynb)

This notebook reproduces **every statistic** quoted in
[The Cooling Gap](https://pointzerofive.net/2026/10/the-cooling-gap),
story #003 on *pointzerofive*. Run it top to bottom (`Runtime → Run all` in Colab).

**The question:** the planet is heating up for everyone — so who actually gets to stay cool?

It works in two layers:

1. **From the published story dataset** (`data/the-cooling-gap.json`): state-level correlations,
   the regression, the hot/poor split and its test.
2. **From the study's raw output files** (`notebooks/raw/pavanello2021/*.dta`, copies of the
   replication files of Pavanello et al. 2021): the decile medians, the 2040 household counts,
   electricity growth and the household-level heat effects — each checked against the dataset.

Every quoted number is printed and `assert`-ed, so a silent change in the data breaks the run.

In [ ]:
import io
import json
import math
import pathlib
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats

BASE = "https://raw.githubusercontent.com/john-bruene/pointzerofive-data/main/the-cooling-gap/"
REPO = pathlib.Path("..")  # notebook lives in notebooks/ when run inside the repo


def fetch(rel_path: str) -> bytes:
    # next to the notebook (pointzerofive-data), in the website's repository, else downloaded
    name = rel_path.rsplit("/", 1)[-1]
    for local in (pathlib.Path(name), REPO / rel_path):
        if local.exists():
            return local.read_bytes()
    with urllib.request.urlopen(BASE + name) as response:
        return response.read()


def read_dta(name: str) -> pd.DataFrame:
    return pd.read_stata(io.BytesIO(fetch(f"notebooks/raw/pavanello2021/{name}")))


def close(a, b, tol=0.051):
    assert math.isclose(a, b, abs_tol=tol), (a, b)
    return True


raw = json.loads(fetch("data/the-cooling-gap.json"))
S = raw["stats"]
states = pd.DataFrame(raw["states"])
print(f"{len(states)} states, {states.householdsM.sum():.1f} million households")
states.head()

## The measures

* **Heat exposure:** cooling degree days (CDD) on wet-bulb temperature, 24 °C base — for each day,
  how far the humidity-adjusted temperature climbs above 24 °C, summed over the year. State means,
  1970–2016.
* **Spending:** household expenditure per person in 2011 US dollars (the study's income proxy).
* **AC share:** survey-weighted share of households owning an air conditioner
  (India NSS 2012, Mexico ENIGH 2016, Brazil POF 2017).

## Part 1 — heat or money? Correlations within each country

In [ ]:
states["lspend"] = np.log(states.spendPc)
rows = []
for country, g in states.groupby("country"):
    r_heat, p_heat = stats.pearsonr(g.cdd, g.ac)
    r_spend, p_spend = stats.pearsonr(g.lspend, g.ac)
    rows.append({"country": country, "n": len(g), "r_heat": r_heat, "p_heat": p_heat,
                 "r_spend": r_spend, "p_spend": p_spend})
    close(round(r_heat, 2), S["by_country"][country]["heat"]["r"], 0.006)
    close(round(r_spend, 2), S["by_country"][country]["spend"]["r"], 0.006)
corr = pd.DataFrame(rows).set_index("country")
corr.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, col, label in [(axes[0], "cdd", "Heat exposure (wet-bulb CDD per year)"),
                       (axes[1], "spendPc", "Spending per person (2011 US$, log scale)")]:
    for country, shade in [("India", "#222222"), ("Brazil", "#777777"), ("Mexico", "#bbbbbb")]:
        g = states[states.country == country]
        ax.scatter(g[col], g.ac, s=8 + g.householdsM * 12, color=shade, alpha=0.75, label=country)
    ax.set_xlabel(label)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_xscale("log")
axes[0].set_ylabel("Households with AC (%)")
axes[0].legend(frameon=False)
fig.suptitle("Figure 1 — AC share by state against heat and spending", x=0.02, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

**Our hypothesis, written before the analysis:** income predicts AC far better than heat.
To compare the two fairly across countries, we standardize every variable *within* its country
and regress AC share on both.

In [ ]:
z = states.copy()
for v in ["cdd", "lspend", "ac"]:
    z[v] = z.groupby("country")[v].transform(lambda x: (x - x.mean()) / x.std())
m = smf.ols("ac ~ cdd + lspend", data=z).fit()
mi = smf.ols("ac ~ cdd * lspend", data=z).fit()
print(m.summary().tables[1])
print(f"R² = {m.rsquared:.2f}")
print(f"interaction β = {mi.params['cdd:lspend']:.2f}, p = {mi.pvalues['cdd:lspend']:.3f}")

R = S["regression"]
close(round(m.params["cdd"], 2), R["beta_heat"], 0.006)
close(round(m.params["lspend"], 2), R["beta_spend"], 0.006)
close(round(m.rsquared, 2), R["r2"], 0.006)
close(round(mi.params["cdd:lspend"], 2), R["beta_interaction"], 0.006)
close(mi.pvalues["cdd:lspend"], R["p_interaction"], 0.0005)
assert m.pvalues["cdd"] < 0.001 and m.pvalues["lspend"] < 0.001
print("✓ heat and spending tie (β ≈ 0.41 each); the hypothesis was half wrong")

## The hot-and-poor corner

Split each country's states at the **median** on heat and on spending, then compute the
household-weighted AC share in each quadrant. Test hot-poor against hot-rich states with a
two-sided Mann–Whitney U test (no normality assumption, n = 27 vs 18).

In [ ]:
states["hot"] = states.groupby("country").cdd.transform(lambda x: x > x.median())
states["rich"] = states.groupby("country").spendPc.transform(lambda x: x > x.median())
assert (states.hot == pd.DataFrame(raw["states"]).hot).all()
assert (states.rich == pd.DataFrame(raw["states"]).rich).all()

quad = (states.groupby(["hot", "rich"])
        .apply(lambda g: pd.Series({"states": len(g),
                                    "households_m": g.householdsM.sum(),
                                    "ac_share": np.average(g.ac, weights=g.householdsM)}),
               include_groups=False)
        .round(1))
print(quad)

for (hot, rich), key in [((True, False), "hot_poor"), ((True, True), "hot_rich"),
                         ((False, False), "cool_poor"), ((False, True), "cool_rich")]:
    close(quad.loc[(hot, rich), "ac_share"], S["quadrants"][key]["ac_share"], 0.15)
    close(quad.loc[(hot, rich), "households_m"], S["quadrants"][key]["households_m"], 0.15)

mw = stats.mannwhitneyu(states[states.hot & ~states.rich].ac, states[states.hot & states.rich].ac,
                        alternative="two-sided")
print(f"Mann–Whitney p = {mw.pvalue:.3f}")
close(mw.pvalue, S["hot_poor_vs_hot_rich_mannwhitney_p"], 0.002)

Small differences in the quadrant shares (≤ 0.1 point) come from household counts being
rounded to thousands in the published dataset; the pipeline uses unrounded counts.

## Part 2 — inside the household

The study fits logit models of AC ownership on household microdata (not public for all
countries), with heat, spending and their interaction plus household controls. Its published
margins give (a) Pr(AC) against heat for an average household and (b) the effect of heat by
spending level. We read them straight from the raw files.

In [ ]:
SURVEY = {"India": "NSS", "Brazil": "POF", "Mexico": "ENIGH"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for country, shade in [("India", "#222222"), ("Brazil", "#777777"), ("Mexico", "#aaaaaa")]:
    curve = read_dta(f"{SURVEY[country]}_ac_CDD_wb_pred_prob.dta")
    axes[0].plot(curve._at1, curve._margin * 100, color=shade, label=country)
    axes[0].fill_between(curve._at1, curve._ci_lb * 100, curve._ci_ub * 100, color=shade, alpha=0.12)
    first, last = raw["heatCurve"][country][0], raw["heatCurve"][country][-1]
    close(round(curve._margin.iloc[0] * 100, 1), first["p"])
    close(round(curve._margin.iloc[-1] * 100, 1), last["p"])

    eff = read_dta(f"{SURVEY[country]}_ac_CDD_wb_exp.dta")
    eff = eff[(eff._at2 >= 6) & (eff._at2 <= 12)]
    pp, lo, hi = eff._margin * 1e4, eff._ci_lb * 1e4, eff._ci_ub * 1e4  # points per +100 CDD
    axes[1].plot(np.exp(eff._at2), pp, color=shade, label=country)
    axes[1].fill_between(np.exp(eff._at2), lo, hi, color=shade, alpha=0.12)

axes[0].set(xlabel="Heat exposure (wet-bulb CDD per year)", ylabel="Probability of owning AC (%)")
axes[1].set(xscale="log", xlabel="Household spending (2011 US$ per year)",
            ylabel="AC probability gain per +100 CDD (points)")
axes[1].axhline(0, color="#999999", lw=0.8, ls="--")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
fig.suptitle("Figure 2 — heat matters on average, but only above a spending threshold", x=0.02,
             ha="left", fontsize=11)
plt.tight_layout()
plt.show()

india = read_dta("NSS_ac_CDD_wb_exp.dta").set_index("_at2")
poor, rich = india.loc[7.0], india.loc[11.0]
print(f"India, ~${math.exp(7):,.0f}/yr: {poor._margin * 1e4:.2f} points per +100 CDD "
      f"(95% CI {poor._ci_lb * 1e4:.2f} to {poor._ci_ub * 1e4:.2f})")
print(f"India, ~${math.exp(11):,.0f}/yr: {rich._margin * 1e4:.1f} points per +100 CDD")
close(round(poor._margin * 1e4, 2), 0.06, 0.006)
close(round(rich._margin * 1e4, 1), 7.2)

# "No detectable effect": spending levels where the 95% CI includes zero in India or Mexico
for country in ["India", "Mexico"]:
    e = read_dta(f"{SURVEY[country]}_ac_CDD_wb_exp.dta")
    e = e[(e._at2 >= 6) & (e._at2 <= 12)]
    top = e.loc[e._ci_lb <= 0, "_at2"].max()
    print(f"{country}: CI includes zero up to ~${math.exp(top):,.0f} per year")
assert max(read_dta(f"{SURVEY[c]}_ac_CDD_wb_exp.dta").query("_at2 >= 6 and _ci_lb <= 0")._at2.max()
           for c in ["India", "Mexico"]) == 8.0  # e^8 ≈ $3,000

## Part 3 — 2040 by spending decile

Households are ranked by spending **nationally** and cut into ten deciles. The study's file
gives the mean AC share per state × decile; following its Figure 5, we summarize each decile
by the **median across states**. The 2040 projections combine two emission pathways
(RCP 4.5, 8.5) with five Shared Socioeconomic Pathways (SSP1–5). The middle-of-the-road
scenario is RCP 4.5 / SSP2.

In [ ]:
dec = read_dta("HH_4countries_proj_dec.dta")
SCEN = [f"{r}_{s}" for r in ["rcp45", "rcp85"] for s in ["SSP1", "SSP2", "SSP3", "SSP4", "SSP5"]]
MAIN = raw["meta"]["mainScenario"]
med = dec.groupby(["country", "dec_inc"])[["ac"] + [f"ac_fut_{s}" for s in SCEN]].median() * 100

summary = pd.DataFrame({
    "today": med.ac,
    "2040 main": med[f"ac_fut_{MAIN}"],
    "2040 min": med[[f"ac_fut_{s}" for s in SCEN]].min(axis=1),
    "2040 max": med[[f"ac_fut_{s}" for s in SCEN]].max(axis=1),
}).round(1)

for country in ["India", "Brazil", "Mexico"]:
    for row in raw["deciles"]["series"][country]:
        got = summary.loc[(country, row["decile"])]
        close(got["today"], row["today"]); close(got["2040 main"], row["main"])
        close(got["2040 min"], row["lo"]); close(got["2040 max"], row["hi"])
summary.unstack(0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, country in zip(axes, ["India", "Brazil", "Mexico"]):
    t = summary.loc[country]
    ax.fill_between(t.index, t["2040 main"], 100, color="#c0392b", alpha=0.15, label="still without AC")
    ax.fill_between(t.index, t["2040 min"], t["2040 max"], color="#000000", alpha=0.08, label="10 scenarios")
    ax.plot(t.index, t["today"], "o--", color="#999999", label="today")
    ax.plot(t.index, t["2040 main"], "o-", color="#222222", label="2040, RCP 4.5 / SSP2")
    ax.set_title(country, loc="left")
    ax.set_xlabel("Spending decile (1 = poorest)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Households with AC (%, median of states)")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Figure 3 — growth fills the top of the distribution first", x=0.02, ha="left", fontsize=11)
plt.tight_layout()
plt.show()

## Part 4 — how many households?

The study's state file counts households with and without AC today and in each 2040 scenario
(millions). Summing over states:

In [ ]:
f4 = read_dta("Figure_4.dta")
W = S["without_ac"]
today = f4.n_without_ac.sum()
households = f4.hh_state.sum() / 1e6
totals = pd.Series({s: f4[f"n_without_ac_{s}"].sum() for s in SCEN}).round(1)
print(f"today: {today:.1f}M of {households:.1f}M households ({today / households:.1%}) without AC")
print(f"India's share of today's total: {f4.loc[f4.country == 'India', 'n_without_ac'].sum() / today:.0%}")
print(totals.sort_values().to_string())
close(today, W["today_m"]); close(households, W["households_today_m"])
close(today / households * 100, W["today_share"])
close(totals[MAIN], W["main_m"]); close(totals.min(), W["min_m"]); close(totals.max(), W["max_m"])
assert totals.idxmin() == W["min_scenario"] and totals.idxmax() == W["max_scenario"]

# "The hotter future has fewer households without AC": RCP 8.5 < RCP 4.5 for every SSP
for ssp in ["SSP1", "SSP2", "SSP3", "SSP4", "SSP5"]:
    assert totals[f"rcp85_{ssp}"] < totals[f"rcp45_{ssp}"], ssp
print("✓ under every SSP, the high-warming pathway leaves fewer households without AC")

### But… not everyone needs AC

A household without AC on a cool plateau has no deficit. Keep only states whose **projected**
(2021–2060) wet-bulb CDD reaches a threshold. The story uses 100; change `THRESHOLDS` to test
other cut-offs.

In [ ]:
THRESHOLDS = [0, 50, 100, 150, 200, 300]


def hot_count(threshold: float, scenario: str) -> float:
    rcp, ssp = scenario.split("_")
    cdd = f4[f"mean_CDD_2021_2060_wb_{rcp}_{ssp}"]
    return f4.loc[cdd >= threshold, f"n_without_ac_{scenario}"].sum()


table = pd.DataFrame({t: {"main": hot_count(t, MAIN),
                          "min": min(hot_count(t, s) for s in SCEN),
                          "max": max(hot_count(t, s) for s in SCEN)} for t in THRESHOLDS}).T.round(1)
table.index.name = "CDD threshold"
print(table)

T = raw["meta"]["hotThresholdCdd"]
close(hot_count(T, MAIN), W["hot_main_m"])
close(min(hot_count(T, s) for s in SCEN), W["hot_min_m"])
close(max(hot_count(T, s) for s in SCEN), W["hot_max_m"])
rcp, ssp = MAIN.split("_")
india_hot = f4.loc[(f4.country == "India") & (f4[f"mean_CDD_2021_2060_wb_{rcp}_{ssp}"] >= T),
                   f"n_without_ac_{MAIN}"].sum()
print(f"India's share of hot-state households without AC ({MAIN}): {india_hot / hot_count(T, MAIN):.0%}")
close(india_hot / hot_count(T, MAIN) * 100, W["hot_main_india_share"], 0.15)

### But… more AC is its own problem

The same projections model household electricity use. Median growth by decile, today → 2040
(middle scenario):

In [ ]:
ely = dec.groupby(["country", "dec_inc"])[["ely_q", f"ely_hat_fut_int_{MAIN}_q"]].median()
growth = ((ely[f"ely_hat_fut_int_{MAIN}_q"] / ely.ely_q - 1) * 100).round(1).unstack(0)
print(growth)
print(f"India: +{growth.India.min():.0f}% to +{growth.India.max():.0f}%; "
      f"Brazil up to +{growth.Brazil.max():.0f}%; Mexico up to +{growth.Mexico.max():.0f}%")
for country in ["India", "Brazil", "Mexico"]:
    for row in raw["electricity"][country]:
        close(growth.loc[row["decile"], country], row["growth"])

Context cited in the story (not computed here): the IEA's *The Future of Cooling* (2018)
projects global energy demand for air-conditioning to triple by 2050; Raman et al.
(*Nature* 515, 540–544, 2014) demonstrated passive radiative cooling 4.9 °C below ambient air
temperature in direct sunlight.

## Data and methods

* **Source.** Pavanello, F. *et al.* Air-conditioning and the adaptation cooling deficit in
  emerging economies. *Nature Communications* 12, 6460 (2021).
  [doi:10.1038/s41467-021-26592-2](https://doi.org/10.1038/s41467-021-26592-2).
  Replication files: [github.com/Energy-a/comparative_paper_natcomms](https://github.com/Energy-a/comparative_paper_natcomms)
  (commit `69a0125`), distributed under the **GNU GPL v3.0**; the copies in
  `notebooks/raw/pavanello2021/` are unmodified and redistributed under the same license.
  The paper is open access (CC BY 4.0).
* **Coverage.** India (NSS 2012), Mexico (ENIGH 2016), Brazil (POF 2017). Indonesia, part of the
  original study, has no public microdata. One state without a climate value is excluded (92 remain).
* **Methods.** Within-country Pearson correlations; OLS on within-country standardized variables;
  household-weighted quadrant shares with a two-sided Mann–Whitney U test; the study's logit
  margins with 95% CIs; decile medians across states (study Figure 5); state sums of households
  without AC (study Figure 4).
* **Caveats.** Ownership is not use; survey years differ; state correlations are ecological;
  the 2040 figures are scenario projections, not forecasts.
* **Pipeline.** `notebooks/cooling_gap_pipeline.py` builds `data/the-cooling-gap.json` from the raw files.

In [ ]:
print("All statistics reproduced ✓")